# Exp 8 — three frozen checks for H-adapter
Put this notebook next to `exp8_lib.py` and `./exp8/`. Everything below is plain cells (no files written); your running jobs are untouched.

| Check | Question | PASS | FAIL |
|---|---|---|---|
| **Gate** | does the instrument see sharing? | mean gain_A′ ≥ 0.02 nats and ≥ 3 SE | → every check is UNTESTABLE, not negative |
| **1. Dissociation** | does rev_all share better than rev4 / rev45? (string distance says no) | s(rev_all) − max(s(rev4), s(rev45)) ≥ 0.10 in every seed (3 seeds) | mean difference ≤ 0 |
| **2. Beats string distance** | does a(g) predict s(g) after controlling for tau and displacement? | partial ρ ≤ −0.5, ρ(a,s) ≤ ρ(tau,s) − 0.10, leave-one-out partial never > −0.3 (3 seeds, 8+ renderings) | partial ≥ −0.2 |
| **3. Depth** | does sharing start at later layers as a(g) rises? | Spearman(a, onset layer) ≥ +0.7 and partial given tau ≥ +0.5 (6+ renderings with an onset) | Spearman ≤ +0.3 |

Between PASS and FAIL = INCONCLUSIVE. **Run the freeze cell (section 2) once, before reading any joint result.** Check 3 is probe transfer, not causal patching, and its torch part was not run by its author: do the one-rendering smoke test first.

## 0. Definitions: checks 1 and 2

In [1]:
# Checks 1 and 2 (CPU only). Defines write_prereg, check1, check2 and helpers.
import os, sys, json, glob, random, time
import numpy as np
import exp8_lib as X

# ----------------------------------------------------------------------------- frozen thresholds (edit only BEFORE joint results exist)
GATE_MIN_GAIN, GATE_Z = 0.02, 3.0        # same instrument gate as the notebook
C1_MARGIN        = 0.10                  # s(rev_all) - max(s(rev4), s(rev45)) must be >= this in EVERY seed
C1_MIN_SEEDS     = 3                     # fewer seeds -> no verdict
C2_MIN_N         = 8                     # renderings with finite a and s
C2_MIN_SEEDS     = 3                     # fewer seeds -> numbers are printed but there is no verdict
C2_PARTIAL_PASS  = -0.5                  # partial rho(a, s | tau, displacement) must be <= this
C2_PARTIAL_DEAD  = -0.2                  # ... and is DEAD if >= this
C2_LOO_MAX       = -0.3                  # leave-one-rendering-out partial must never exceed this
C2_BEAT_MARGIN   = 0.10                  # rho(a, s) must be at least this much more negative than rho(tau, s)

PREREG = f"""H-adapter pre-registration - written {{when}}
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= {GATE_MIN_GAIN} nats and >= {GATE_Z} SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= {C1_MIN_SEEDS} seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= {C1_MARGIN} in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= {C2_MIN_N} renderings with finite a and s, and >= {C2_MIN_SEEDS} seeds.
  PASS         partial rho(a, s | tau, displacement) <= {C2_PARTIAL_PASS}
               AND rho(a, s) <= rho(tau, s) - {C2_BEAT_MARGIN}
               AND the leave-one-rendering-out partial never exceeds {C2_LOO_MAX}
  FAIL         partial >= {C2_PARTIAL_DEAD}
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.
"""

def write_prereg(root):
    path = os.path.join(root, "prereg_three_checks.txt")
    if os.path.exists(path): print("already frozen:", path); print(open(path).read()); return
    os.makedirs(root, exist_ok=True); open(path, "w").write(PREREG.format(when=time.strftime("%Y-%m-%d %H:%M"))); print("wrote", path)

# ----------------------------------------------------------------------------- loading (same definitions as the notebook)
def load_all(root):
    J, T = {}, {}
    for p in glob.glob(os.path.join(root, "joint", "*.json")):
        r = json.load(open(p)); J[(r["g"], r["cond"], r["d"], r["seed"])] = r
    for p in glob.glob(os.path.join(root, "translator", "*.json")):
        r = json.load(open(p)); T[(r["g"], r["d"], r["seed"])] = r
    return J, T

def gains(J, d, seeds):
    out = {}
    for g in X.G_LIST:
        for s in seeds:
            a, b = J.get((g, "shared", d, s)), J.get((g, "disjoint", d, s))
            if a is None or b is None: continue
            gain = {L: b["ce"][L] - a["ce"][L] for L in X.LANGS3}
            se = {L: (a["ce_se"][L] ** 2 + b["ce_se"][L] ** 2) ** 0.5 for L in X.LANGS3}
            ok = gain["A2"] > max(GATE_MIN_GAIN, GATE_Z * se["A2"])
            out[(g, s)] = dict(gain=gain, se=se, anchor_ok=ok, s=gain["G"] / gain["A2"] if ok else float("nan"))
    return out

def cost_a(T, d, seeds):
    out = {}
    for g in X.G_LIST:
        v = [T[(g, d, s)]["auc"] - T[("identity", d, s)]["auc"] for s in seeds if (g, d, s) in T and ("identity", d, s) in T]
        if v: out[g] = float(np.mean(v))
    return out

def gate_status(G):
    if not G: return False, "no joint runs yet"
    anchors = [v["gain"]["A2"] for v in G.values()]; ses = [v["se"]["A2"] for v in G.values()]
    m = float(np.mean(anchors)); sem = max(float(np.mean(ses)) / len(anchors) ** 0.5, 1e-9)
    ok = m >= GATE_MIN_GAIN and m >= GATE_Z * sem
    return ok, f"mean gain_A' {m:.4f} nats over {len(anchors)} runs (SE of mean ~{sem:.4f})"

# ----------------------------------------------------------------------------- stats helpers
def _ranks(x):
    x = np.asarray(x, float); order = np.argsort(x); r = np.empty(len(x)); i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]: j += 1
        r[order[i:j + 1]] = (i + j) / 2 + 1; i = j + 1
    return r

def spearman(x, y):
    rx, ry = _ranks(x), _ranks(y)
    if rx.std() < 1e-12 or ry.std() < 1e-12: return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def _resid(v, Zr): return v - Zr @ np.linalg.lstsq(Zr, v, rcond=None)[0]

def partial_rank_corr(x, y, Z):
    """Spearman of x and y after regressing both ranks on the ranks of every column in Z (Z: list of vectors)."""
    n = len(x); Zr = np.column_stack([np.ones(n)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    if ex.std() < 1e-12 or ey.std() < 1e-12: return float("nan")
    return float(np.corrcoef(ex, ey)[0, 1])

def perm_p_partial(x, y, Z, obs, n=20000, seed=0):
    """one-sided permutation p for partial <= obs, permuting the residual of y (approximate; n is tiny, read it as a rough guide)."""
    rng = np.random.RandomState(seed); m = len(x); Zr = np.column_stack([np.ones(m)] + [_ranks(z) for z in Z])
    ex, ey = _resid(_ranks(x), Zr), _resid(_ranks(y), Zr)
    return float(np.mean([np.corrcoef(ex, rng.permutation(ey))[0, 1] <= obs for _ in range(n)]))

# ----------------------------------------------------------------------------- string baselines (same as the notebook cell)
def _uniq(node, c):
    if isinstance(node, int): c[0] += 1; return 1000 + c[0] - 1
    return (node[0], [_uniq(ch, c) for ch in node[1]])

def string_baselines(g, n_trees=300, seed=3):
    rules, _ = X.build_grammar(X.GRAMMAR_SEED); rng = random.Random(seed); spec = X.TRANSFORMS[g]; disp, tau = [], []
    for _ in range(n_trees):
        t = _uniq(X.sample_tree(rules, rng), [0]); out = X.render(t, spec)
        leaves = np.array([x - 1000 for x in out if x >= 1000]); n = len(leaves); pos = np.empty(n, int); pos[leaves] = np.arange(n)
        disp.append(np.abs(pos - np.arange(n)).mean() / n)
        tau.append(((pos[:, None] > pos[None, :]) & (np.arange(n)[:, None] < np.arange(n)[None, :])).sum() / (n * (n - 1) / 2))
    return dict(displacement=float(np.mean(disp)), kendall_tau=float(np.mean(tau)))

# ----------------------------------------------------------------------------- CHECK 1
def check1(root, d=32, seeds=(0, 7, 13), verbose=True):
    J, T = load_all(root); G = gains(J, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 1: rev_all vs rev4 / rev45  (d={d}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate). Not a negative result. Try smaller d."); return dict(verdict="UNTESTABLE")
    need = ("rev_all", "rev4", "rev45"); diffs = []; sec = []
    print(f"\n{'seed':>5s} {'s(rev_all)':>10s} {'s(rev4)':>8s} {'s(rev45)':>9s} {'diff':>7s}   perm_sym s / rank among all")
    for s in seeds:
        v = {g: G[(g, s)]["s"] for g in X.G_LIST if (g, s) in G}
        if not all(g in v and np.isfinite(v[g]) for g in need): print(f"{s:>5d}   missing or gated-out runs"); continue
        diff = v["rev_all"] - max(v["rev4"], v["rev45"]); diffs.append(diff)
        fin = sorted([(x, g) for g, x in v.items() if np.isfinite(x)]); low2 = [g for _, g in fin[:2]]
        sec.append("perm_sym" in low2)
        print(f"{s:>5d} {v['rev_all']:10.3f} {v['rev4']:8.3f} {v['rev45']:9.3f} {diff:+7.3f}   {v.get('perm_sym', float('nan')):.3f} / {[g for _, g in fin].index('perm_sym') + 1 if 'perm_sym' in v and np.isfinite(v['perm_sym']) else '-'} of {len(fin)}")
    if len(diffs) < C1_MIN_SEEDS:
        print(f"\nVERDICT: NO VERDICT YET - {len(diffs)} usable seed(s), need {C1_MIN_SEEDS}. (Diffs so far: {[round(x, 3) for x in diffs]}.)"); return dict(verdict="NO VERDICT", diffs=diffs)
    if all(x >= C1_MARGIN for x in diffs): v = "PASS"
    elif float(np.mean(diffs)) <= 0: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"\nmean diff {np.mean(diffs):+.3f}, min {min(diffs):+.3f}, max {max(diffs):+.3f}   |   perm_sym among two lowest in {sum(sec)}/{len(sec)} seeds (secondary)")
    print(f"VERDICT: {v}" + ("  <- the dissociation holds; a(g) is not just string distance" if v == "PASS" else "  <- rev_all does NOT share better; a(g) loses its main argument" if v == "FAIL" else ""))
    return dict(verdict=v, diffs=diffs)

# ----------------------------------------------------------------------------- CHECK 2
def check2(root, d=32, seeds=(0, 7, 13), n_boot=2000):
    J, T = load_all(root); G = gains(J, d, list(seeds)); A = cost_a(T, d, list(seeds)); ok, msg = gate_status(G)
    print(f"=== CHECK 2: does a(g) beat string distance?  (d={d}, seeds={list(seeds)}) ===\nGATE: {msg} -> {'PASS' if ok else 'FAIL'}")
    if not ok: print("VERDICT: UNTESTABLE (gate)."); return dict(verdict="UNTESTABLE")
    gs, a, s = [], [], []
    for g in X.G_LIST:
        v = [G[(g, sd)]["s"] for sd in seeds if (g, sd) in G]; v = [x for x in v if np.isfinite(x)]
        if g in A and v: gs.append(g); a.append(A[g]); s.append(float(np.mean(v)))
    if len(gs) < C2_MIN_N: print(f"VERDICT: NO VERDICT YET - {len(gs)} renderings with finite a and s, need {C2_MIN_N}."); return dict(verdict="NO VERDICT")
    B = {g: string_baselines(g) for g in gs}; tau = [B[g]["kendall_tau"] for g in gs]; disp = [B[g]["displacement"] for g in gs]
    r_as, r_ts, r_ds = spearman(a, s), spearman(tau, s), spearman(disp, s)
    p_both = partial_rank_corr(a, s, [tau, disp]); p_tau = partial_rank_corr(a, s, [tau]); p_disp = partial_rank_corr(a, s, [disp])
    loo = [partial_rank_corr([x for j, x in enumerate(a) if j != i], [x for j, x in enumerate(s) if j != i], [[x for j, x in enumerate(z) if j != i] for z in (tau, disp)]) for i in range(len(gs))]
    rng = np.random.RandomState(1); boots = []
    for _ in range(n_boot):
        ix = rng.randint(0, len(gs), len(gs)); b = partial_rank_corr(np.array(a)[ix], np.array(s)[ix], [np.array(tau)[ix], np.array(disp)[ix]])
        if np.isfinite(b): boots.append(b)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (float("nan"),) * 2)
    print(f"\nn = {len(gs)} renderings: {gs}")
    print(f"rho(a, s)               = {r_as:+.2f}")
    print(f"rho(tau, s)             = {r_ts:+.2f}     rho(displacement, s) = {r_ds:+.2f}   <- the baselines")
    print(f"partial rho(a,s | tau)  = {p_tau:+.2f}     | disp = {p_disp:+.2f}     | tau AND disp = {p_both:+.2f}   (bootstrap 95% CI over renderings [{lo:+.2f}, {hi:+.2f}])")
    print(f"leave-one-out partial   range [{np.nanmin(loo):+.2f}, {np.nanmax(loo):+.2f}]   perm p (partial <= observed) = {perm_p_partial(a, s, [tau, disp], p_both):.3f}")
    beats = r_as <= r_ts - C2_BEAT_MARGIN
    n_seeds = len({sd for (_, sd) in G})
    if n_seeds < C2_MIN_SEEDS: print(f"\nVERDICT: NO VERDICT YET - {n_seeds} seed(s) of joint results, need {C2_MIN_SEEDS}. The numbers above are provisional."); return dict(verdict="NO VERDICT", rho_as=r_as, partial=p_both)
    if p_both <= C2_PARTIAL_PASS and beats and np.nanmax(loo) <= C2_LOO_MAX: v = "PASS"
    elif p_both >= C2_PARTIAL_DEAD: v = "FAIL"
    else: v = "INCONCLUSIVE"
    print(f"a beats tau as a predictor of s by >= {C2_BEAT_MARGIN}: {beats}")
    print(f"VERDICT: {v}" + ("  <- a(g) carries information string distance does not" if v == "PASS" else "  <- once tau/displacement are controlled, a(g) predicts nothing" if v == "FAIL" else "  <- more renderings (rev3, rev345, rev2, rev23) and seeds before concluding"))
    print("Note: with ~10 renderings the partial has ~7 degrees of freedom; the bootstrap CI above is wide on purpose. Report it.")
    return dict(verdict=v, rho_as=r_as, rho_ts=r_ts, partial=p_both, ci=(float(lo), float(hi)))


## 0b. Definitions: check 3 (depth)

In [2]:
"""exp8_depth.py - Check 3: does cross-rendering sharing start LATER in depth as conversion cost a(g) rises?

Readout (a cheap proxy for the Exp 7 patching; it is a probe, not a causal intervention):
  every token of a rendering has a ground-truth ancestor symbol at each tree level, whatever order the tokens are written in.
  Train a linear probe for the level-k ancestor on canonical-language (A) residual states at layer l, apply it unchanged to rendering-g states.
  T_l(g) = (acc[A-probe on g] - chance) / (acc[g-probe on g] - chance)
  The denominator is g's OWN ceiling (probe trained on g), not A's: reordered strings put different information in each prefix, so A's accuracy is not g's ceiling.
  onset(g) = first layer from which T stays >= ONSET_FRAC to the last layer; NaN if never.

Frozen prediction (edit only before the first run):
  Primary: PRIMARY_LEVEL = 4, centered probes, median over seeds.
  PASS  Spearman(a, onset) >= +0.7 over renderings with a defined onset, AND partial given tau >= +0.5
  FAIL  Spearman(a, onset) <= +0.3
  INCONCLUSIVE otherwise.  UNTESTABLE if fewer than 6 renderings reach ONSET_FRAC, or if the shared-model gate has not passed.
  Reported but not part of the verdict: other levels, strict (uncentered) probes, mean T over layers, an untrained-model floor.
"""
import os, sys, json, random, argparse
import numpy as np
import exp8_lib as X

LEVELS = X.LEVELS
PROBE_LEVELS = [2, 3, 4, 5, 6]          # level 1 has a single symbol, so nothing to decode
PRIMARY_LEVEL = 4
ONSET_FRAC = 0.5
MIN_CEIL_GAP = 0.05                     # layers where the g-probe barely beats chance on g are masked (T undefined)
N_TRAIN, N_TEST = 400, 400              # trees per language per split
TRAIN_SEED, TEST_SEED = 31, 32
C3_PASS_RHO, C3_PASS_PARTIAL, C3_FAIL_RHO, C3_MIN_DEFINED = 0.7, 0.5, 0.3, 6

# ----------------------------------------------------------------------------- rows with ancestor labels (torch-free)
def render_lab(node, spec, anc=()):
    """Mirror of X.render that also returns, per output token, the ancestor symbol index at levels 1..LEVELS.
    A terminal's ancestors are its LEVELS enclosing nonterminals; the marker token belongs to the level-LEVELS node that emitted it."""
    if isinstance(node, int): return [spec.get("term_map", {}).get(node, node)], [anc]
    (lvl, idx), children = node; anc2 = anc + (idx,)
    order = list(range(len(children)))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and len(order) >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), len(children))]
    toks, labs = [], []
    for j in order:
        t, l = render_lab(children[j], spec, anc2); toks.extend(t); labs.extend(l)
    if spec.get("marker") is not None and lvl == LEVELS: toks.append(spec["marker"]); labs.append(anc2)
    return toks, labs

def make_rows_lab(grammar, spec, bos, n_trees, seed, max_leaves=160, row_len=X.SEQ_LEN):
    """Same tree sampling as X.make_eval_rows (so the same seed gives the same trees in every language), plus labels [n, row_len, LEVELS] (-1 = no label)."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); nreal = np.zeros(n_trees, np.int64); labs = -np.ones((n_trees, row_len, LEVELS), np.int8); k = 0
    while k < n_trees:
        t = X.sample_tree(grammar, rng)
        if len(X.render(t, {})) > max_leaves: continue
        toks, lab = render_lab(t, spec); assert toks == X.render(t, spec), "render_lab diverged from X.render"
        row = [bos] + toks; assert len(row) <= row_len
        rows[k, :len(row)] = row; nreal[k] = len(row); labs[k, 1:len(row)] = np.asarray(lab, np.int8); k += 1
    return rows, nreal, labs

# ----------------------------------------------------------------------------- probes (numpy)
def _ridge_fit(Xtr, y, ncls, lam=1e-2):
    Xb = np.concatenate([Xtr, np.ones((len(Xtr), 1))], 1); Y = np.eye(ncls)[y]
    return np.linalg.solve(Xb.T @ Xb + lam * len(Xb) * np.eye(Xb.shape[1]), Xb.T @ Y)

def _ridge_acc(W, Xte, y): return float((np.argmax(np.concatenate([Xte, np.ones((len(Xte), 1))], 1) @ W, 1) == y).mean())

def probe_transfer(S, Y, ncls=4, centered=True):
    """S[lang][split]: array [L, N, d]; Y[lang][split]: int array [N] (languages 'A', 'A2', 'G'; splits 'train', 'test').
    Returns per-layer dict: acc_AA, acc_AA2, acc_AG, acc_GG, chance, T (NaN where g's own ceiling is within MIN_CEIL_GAP of chance)."""
    L = S["A"]["train"].shape[0]; out = {k: [] for k in ("acc_AA", "acc_AA2", "acc_AG", "acc_GG", "chance", "T")}
    chance = float(np.bincount(Y["G"]["test"], minlength=ncls).max() / len(Y["G"]["test"]))
    for l in range(L):
        mu = {lg: S[lg]["train"][l].mean(0) for lg in S} if centered else {lg: S["A"]["train"][l].mean(0) for lg in S}
        f = lambda lg, sp: S[lg][sp][l] - mu[lg]
        WA = _ridge_fit(f("A", "train"), Y["A"]["train"], ncls); WG = _ridge_fit(f("G", "train"), Y["G"]["train"], ncls)
        aa, aa2 = _ridge_acc(WA, f("A", "test"), Y["A"]["test"]), _ridge_acc(WA, f("A2", "test"), Y["A2"]["test"])
        ag, gg = _ridge_acc(WA, f("G", "test"), Y["G"]["test"]), _ridge_acc(WG, f("G", "test"), Y["G"]["test"])
        T = (ag - chance) / (gg - chance) if gg - chance >= MIN_CEIL_GAP else float("nan")
        for k, v in zip(out, (aa, aa2, ag, gg, chance, T)): out[k].append(float(v))
    return out

def onset_layer(T, frac=ONSET_FRAC):
    """start of the final unbroken run of layers with T >= frac that reaches the last layer. A layer where T is undefined (g's own probe cannot decode yet)
    breaks the run: 'cannot be measured' is not evidence of sharing. NaN if the last layer is below frac or undefined."""
    T = np.asarray(T, float)
    if not np.isfinite(T[-1]) or T[-1] < frac: return float("nan")
    l = len(T) - 1
    while l - 1 >= 0 and np.isfinite(T[l - 1]) and T[l - 1] >= frac: l -= 1
    return float(l)

def decodable_layer(acc_GG, chance, gap=MIN_CEIL_GAP):
    """first layer where g's OWN probe beats chance by `gap`; onset cannot precede it, so onset - decodable is the lag attributable to transfer"""
    a = np.asarray(acc_GG, float) - chance; idx = np.where(a >= gap)[0]
    return float(idx[0]) if len(idx) else float("nan")

# ----------------------------------------------------------------------------- torch part: hidden states at every layer
try:
    import torch
    TORCH = True
except Exception:
    TORCH = False

if TORCH:
    @torch.no_grad()
    def layer_states(model, rows, nreal, labs, dev, bs=64):
        """returns states [L+1, Ntok, d] (embedding output, then every block output) at the rendering tokens (positions 1..nreal-1) and labels [Ntok, LEVELS]"""
        model.eval(); store = []; hs = [model.tok_emb.register_forward_hook(lambda m, i, o: store.append(o))]
        for blk in model.blocks: hs.append(blk.register_forward_hook(lambda m, i, o: store.append(o)))
        S, Ys = [], []; ar = torch.arange(rows.shape[1], device=dev)
        try:
            for i in range(0, rows.shape[0], bs):
                x = torch.from_numpy(rows[i:i + bs].astype(np.int64)).to(dev); nr = torch.from_numpy(nreal[i:i + bs]).to(dev); store.clear(); model(x)
                H = torch.stack(store, 0).float()                                   # [L+1, B, T, d]
                m = (ar[None, :] >= 1) & (ar[None, :] < nr[:, None])               # rendering tokens only, BOS and padding excluded
                S.append(H[:, m].cpu().numpy()); Ys.append(labs[i:i + bs][m.cpu().numpy()])
        finally:
            for h in hs: h.remove()
        return np.concatenate(S, 1), np.concatenate(Ys, 0)

    def load_model(root, g, cond, d, seed, dev, n_head=4, n_layer=12, untrained=False):
        if untrained: torch.manual_seed(seed + 1000)
        model = X.TinyRotaryGPT(X.VOCAB, d, n_layer, n_head, X.SEQ_LEN + 1).to(dev)
        if not untrained: model.load_state_dict(torch.load(X.joint_paths(root, g, cond, d, seed)[1], map_location=dev)["model_state"])
        return model

    def run_one(root, g, d, seed, device, untrained=False):
        dev = torch.device(device); path = os.path.join(root, "depth", f"{g}{'_untrained' if untrained else ''}_d{d}_s{seed}.json")
        if os.path.exists(path): print("done already:", path); return
        model = load_model(root, g, "shared", d, seed, dev, untrained=untrained); grams = X.make_grammars("shared"); specs = {"A": {}, "A2": {}, "G": X.TRANSFORMS[g]}
        S, Y = {}, {}
        for lg in ("A", "A2", "G"):
            S[lg], Y[lg] = {}, {}
            for sp, n, sd in (("train", N_TRAIN, TRAIN_SEED), ("test", N_TEST, TEST_SEED)):
                rows, nreal, labs = make_rows_lab(grams[lg], specs[lg], X.BOS[lg], n, sd); S[lg][sp], Y[lg][sp] = layer_states(model, rows, nreal, labs, dev)
        res = dict(g=g, d=d, seed=seed, untrained=untrained, levels={})
        for k in PROBE_LEVELS:
            Yk = {lg: {sp: Y[lg][sp][:, k - 1].astype(int) for sp in Y[lg]} for lg in Y}
            for centered in (True, False):
                r = probe_transfer(S, Yk, centered=centered); r["onset"] = onset_layer(r["T"]); res["levels"][f"{k}_{'centered' if centered else 'strict'}"] = r
        os.makedirs(os.path.dirname(path), exist_ok=True); json.dump(res, open(path, "w")); print("wrote", path, "level-4 centered T by layer:", np.round(res["levels"]["4_centered"]["T"], 2).tolist(), flush=True)

# ----------------------------------------------------------------------------- summary and verdict (numpy only)
def summarize(root, d=32, seeds=(0,), level=PRIMARY_LEVEL, mode="centered"):
    import glob
    key = f"{level}_{mode}"; per = {}
    for g in X.G_LIST:
        runs = [json.load(open(p)) for s in seeds for p in glob.glob(os.path.join(root, "depth", f"{g}_d{d}_s{s}.json"))]
        if runs: per[g] = dict(onset=[r["levels"][key]["onset"] for r in runs], T=np.nanmean([r["levels"][key]["T"] for r in runs], 0), acc_AA2=np.mean([r["levels"][key]["acc_AA2"] for r in runs], 0),
                      dec=decodable_layer(np.mean([r["levels"][key]["acc_GG"] for r in runs], 0), runs[0]["levels"][key]["chance"][0]))
    _, T = load_all(root); A = cost_a(T, d, list(seeds)); gs = [g for g in per if g in A]
    print(f"=== CHECK 3: depth of sharing (level {level}, {mode} probes, d={d}, seeds={list(seeds)}) ===")
    if len(gs) < 4: print("VERDICT: NO VERDICT YET - fewer than 4 renderings have both a(g) and a depth result."); return
    print(f"\n{'g':13s} {'a':>7s} {'onset':>6s} {'decod.':>6s} {'mean T':>7s}  T by layer 0..{len(per[gs[0]]['T']) - 1}   (decod. = first layer where g's own probe works)")
    for g in sorted(gs, key=lambda g: A[g]):
        on = float(np.nanmedian(per[g]["onset"])) if np.isfinite(per[g]["onset"]).any() else float("nan")
        print(f"{g:13s} {A[g]:7.4f} {on:6.1f} {per[g]['dec']:6.1f} {np.nanmean(per[g]['T']):7.2f}  " + " ".join("  . " if not np.isfinite(t) else f"{t:4.1f}" for t in per[g]["T"]))
        per[g]["on"] = on
    ok = [g for g in gs if np.isfinite(per[g]["on"])]; sanity = np.mean([per[g]["acc_AA2"][-1] for g in gs])
    print(f"\nsanity: A-probe on A' (same order, new tag) at the last layer = {sanity:.2f} (should be near the A-on-A accuracy)")
    if len(ok) < C3_MIN_DEFINED: print(f"VERDICT: UNTESTABLE - only {len(ok)} renderings reach T >= {ONSET_FRAC}; fewer than {C3_MIN_DEFINED}. Look at mean T instead, and at the shared-model gate."); return
    a = [A[g] for g in ok]; on = [per[g]["on"] for g in ok]; tau = [string_baselines(g)["kendall_tau"] for g in ok]
    rho, prt = spearman(a, on), partial_rank_corr(a, on, [tau]); rmt = spearman([A[g] for g in gs], [-np.nanmean(per[g]["T"]) for g in gs])
    lag = [per[g]["on"] - per[g]["dec"] for g in ok]; rlag = spearman(a, lag) if len(ok) >= 4 else float("nan")
    print(f"Spearman(a, onset) = {rho:+.2f} (n={len(ok)});  partial given tau = {prt:+.2f};  secondary: Spearman(a, -mean T) = {rmt:+.2f} (n={len(gs)}),  Spearman(a, onset - decodable layer) = {rlag:+.2f}")
    v = "PASS" if (rho >= C3_PASS_RHO and prt >= C3_PASS_PARTIAL) else "FAIL" if rho <= C3_FAIL_RHO else "INCONCLUSIVE"
    print(f"VERDICT: {v}" + ("  <- sharing starts later for costlier renderings, beyond string distance" if v == "PASS" else "  <- depth of sharing does not track a(g)" if v == "FAIL" else ""))
    print("Reminder: this is probe transfer (shared readout), not causal patching. A PASS motivates the patching version; it does not replace it.")


## 1. Config

In [3]:
ROOT, D_MODEL = "./exp8", 32
SEEDS = [0, 7, 13]            # checks 1 and 2 need >= 3 seeds with results
DEVICE = "cuda:2"             # depth run only
print("renderings:", X.G_LIST)

renderings: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym']


## 2. Freeze the criteria (run ONCE, before any joint result is read)

In [4]:
write_prereg(ROOT)

already frozen: ./exp8/prereg_three_checks.txt
H-adapter pre-registration - written 2026-10-29 18:26
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= 0.02 nats and >= 3.0 SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= 3 seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= 0.1 in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= 8 renderings with finite a and s, and >= 3 seeds.
  PASS         partial rho(a, s | tau, displacement) <= -0.5
               AND rho(a, s) <= rho(tau, s) - 0.1
               AND the leave-one-rendering-out partial never exceeds -0.3
  FAIL         partial >= -0.2
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.



## 3. What exists so far

In [5]:
J, TR = load_all(ROOT)
print("T = translator done, S/D = shared / disjoint joint done, Y = shared checkpoint on disk (needed for check 3)\n")
for g in ["identity"] + X.G_LIST:
    row = []
    for s in SEEDS:
        tr = (g, D_MODEL, s) in TR; sh, dj = (g, "shared", D_MODEL, s) in J, (g, "disjoint", D_MODEL, s) in J
        ck = g != "identity" and os.path.exists(X.joint_paths(ROOT, g, "shared", D_MODEL, s)[1])
        row.append(f"s{s:<2d}: {'T' if tr else '.'} {'S' if sh else '.'}{'D' if dj else '.'} {'Y' if ck else '.'}   ")
    print(f"{g:13s} " + "".join(row))

T = translator done, S/D = shared / disjoint joint done, Y = shared checkpoint on disk (needed for check 3)

identity      s0 : T .. .   s7 : T .. .   s13: T .. .   
relabel_new   s0 : T SD Y   s7 : T .. .   s13: T .. .   
relabel_perm  s0 : T SD Y   s7 : T .. .   s13: T .. .   
marker        s0 : T SD Y   s7 : T .. .   s13: T .. .   
swap5         s0 : T SD Y   s7 : T .. .   s13: T .. .   
rev6          s0 : T SD Y   s7 : T .. .   s13: . .. .   
rev5          s0 : T S. Y   s7 : T .. .   s13: . .. .   
rev4          s0 : T .. .   s7 : T .. .   s13: . .. .   
rev45         s0 : T .. .   s7 : T .. .   s13: . .. .   
rev_all       s0 : T .. .   s7 : T .. .   s13: . .. .   
perm_sym      s0 : T .. .   s7 : T .. .   s13: . .. .   


## 4. Checks 1 and 2 (CPU, instant)

In [6]:
r1 = check1(ROOT, d=D_MODEL, seeds=SEEDS)

=== CHECK 1: rev_all vs rev4 / rev45  (d=32) ===
GATE: mean gain_A' -0.0433 nats over 5 runs (SE of mean ~0.0006) -> FAIL
VERDICT: UNTESTABLE (gate). Not a negative result. Try smaller d.


In [7]:
r2 = check2(ROOT, d=D_MODEL, seeds=SEEDS)

=== CHECK 2: does a(g) beat string distance?  (d=32, seeds=[0, 7, 13]) ===
GATE: mean gain_A' -0.0433 nats over 5 runs (SE of mean ~0.0006) -> FAIL
VERDICT: UNTESTABLE (gate).


## 5. Check 3: depth (GPU, forward passes only, so it runs in the kernel)
Smoke test one rendering first and read the printed curves.

In [8]:
run_one(ROOT, "rev4", D_MODEL, 0, DEVICE)
r = json.load(open(f"{ROOT}/depth/rev4_d{D_MODEL}_s0.json"))["levels"]["4_centered"]
print("T by layer :", np.round(r["T"], 2).tolist())
print("A->A' acc  :", np.round(r["acc_AA2"], 2).tolist(), " (same order, new tag: should be close to A->A)")
print("A->A acc   :", np.round(r["acc_AA"], 2).tolist())
print("g->g acc   :", np.round(r["acc_GG"], 2).tolist(), " chance:", round(r["chance"][0], 2))

FileNotFoundError: [Errno 2] No such file or directory: './exp8/joint/rev4_shared_d32_s0.pt'

In [ ]:
!tail -n 8 exp8/scheduler.log
!ps -eo pid,etime,args | grep -E "exp8_lib|exp8_queue" | grep -v grep
!ls exp8/joint 2>/dev/null
!tail -n 2 exp8/logs/jt_rev4_shared_d32_s0.txt

14:44:37 started jt_rev4_disjoint_d32_s0 on cuda:2 pid 3095537
14:44:57 started jt_rev45_shared_d32_s0 on cuda:2 pid 3097011
16:32:25 started jt_rev45_disjoint_d32_s0 on cuda:2 pid 3347480
16:32:56 started jt_rev_all_shared_d32_s0 on cuda:2 pid 3348150
16:33:06 started jt_rev_all_disjoint_d32_s0 on cuda:2 pid 3348444
16:38:38 started jt_perm_sym_shared_d32_s0 on cuda:2 pid 3354358
16:38:58 started jt_perm_sym_disjoint_d32_s0 on cuda:2 pid 3354784
all jobs finished
    torch._C.PyTorchFileWriter(
RuntimeError: Parent directory ./exp8/joint does not exist.


In [ ]:
for s in SEEDS:
    for g in X.G_LIST:
        if not os.path.exists(X.joint_paths(ROOT, g, "shared", D_MODEL, s)[1]): print("no checkpoint yet:", g, s); continue
        run_one(ROOT, g, D_MODEL, s, DEVICE)

In [ ]:
summarize(ROOT, d=D_MODEL, seeds=SEEDS)

## 6. After the verdicts
- Report all three, failures included.
- Check 1 FAIL: a(g) loses its non-obvious argument; tell Anwoy before adding renderings.
- Check 2 INCONCLUSIVE: add rev3, rev345, rev2, rev23 and more seeds before concluding.
- Check 3 UNTESTABLE: read mean T (in the table) and the shared-model gate.

In [ ]:
import os
os.makedirs("exp8/joint", exist_ok=True)

In [ ]:
import os, sys, subprocess
os.makedirs("exp8_smoke/joint", exist_ok=True)
r = subprocess.run([sys.executable, "exp8_lib.py", "joint", "--g", "rev4", "--cond", "shared",
                    "--seed", "0", "--d", "32", "--steps", "300", "--big_n", "5000",
                    "--root", "./exp8_smoke", "--device", "cuda:2"], capture_output=True, text=True)
print(r.stdout[-600:]); print(r.stderr[-1200:])
print(os.listdir("exp8_smoke/joint"))

pool (4645, 257) in 0.0 min
{"g": "rev4", "cond": "shared", "d": 32, "seed": 0, "steps": 300, "tail_loss": 1.096936821937561, "ce": {"A": 0.9052546620368958, "A2": 0.9064743518829346, "G": 0.9100030064582825}, "ce_se": {"A": 0.0012723237741738558, "A2": 0.0012710790615528822, "G": 0.0012654084712266922}, "n_eval": 1000}


['rev4_shared_d32_s0.pt', 'rev4_shared_d32_s0.json']


In [ ]:
import subprocess, sys, os
subprocess.run(["pkill", "-f", "exp8_queue.py"])
if os.path.exists("exp8/scheduler.pid"): os.remove("exp8/scheduler.pid")
cmd = [sys.executable, "exp8_queue.py", "--root", "./exp8", "--d", "32", "--seeds", "0",
       "--devices", "cuda:2", "--max_par", "11", "--joint"]
p = subprocess.Popen(cmd, stdout=open("exp8/scheduler.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
print(p.pid)

3519158


In [ ]:
!ls exp8
!find . -maxdepth 4 \( -name "*_shared_d32_s0.pt" -o -name "*_d32_s0.json" -path "*joint*" \) 2>/dev/null

joint  logs  prereg_three_checks.txt  scheduler.log  scheduler.pid  translator
./exp8_smoke/joint/rev4_shared_d32_s0.pt
./exp8_smoke/joint/rev4_shared_d32_s0.json


In [ ]:
import subprocess, sys, os
import exp8_lib as X
lines = []
for seed in (7, 13):
    for g in ["identity"] + X.G_LIST:
        if not os.path.exists(X.tr_paths("./exp8", g, 32, seed)):
            lines.append(f"{sys.executable} exp8_lib.py translator --g {g} --seed {seed} --d 32 --steps 24000 "
                         f"--root ./exp8 --device cuda:0 > exp8/logs/tr_{g}_d32_s{seed}.txt 2>&1")
open("exp8/tr_extra_jobs.sh", "w").write("\n".join(lines) + "\n")
p = subprocess.Popen("cat exp8/tr_extra_jobs.sh | xargs -P 8 -I{} bash -c '{}'", shell=True,
                     start_new_session=True, stdout=open("exp8/tr_extra.log", "a"), stderr=subprocess.STDOUT)
print(len(lines), "jobs, pid", p.pid)

22 jobs, pid 3531255


In [ ]:
!echo "--- scheduler / jobs alive:"; ps -eo pid,etime,args | grep -E "exp8_queue|exp8_lib|xargs" | grep -v grep | cut -c1-150
!echo "--- scheduler log:"; tail -n 6 exp8/scheduler.log
!echo "--- finished joint results:"; ls exp8/joint 2>/dev/null
!echo "--- progress per joint job (last log line):"; for f in exp8/logs/jt_*_s0.txt; do echo "$f: $(tail -n 1 $f | cut -c1-90)"; done
!echo "--- smoke test:"; ls exp8_smoke/joint 2>/dev/null

--- scheduler / jobs alive:


3519158    02:17:26 /home/anwoy/miniconda3/envs/ke/bin/python exp8_queue.py --root ./exp8 --d 32 --seeds 0 --devices cuda:2 --max_par 11 --joint
3531255    02:08:49 /bin/sh -c cat exp8/tr_extra_jobs.sh | xargs -P 8 -I{} bash -c '{}'
3531257    02:08:49 xargs -P 8 -I{} bash -c {}
3656565       07:47 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev6 --seed 13 --d 32 --steps 24000 --root ./exp8 --d
3656566       07:47 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev6 --seed 13 --d 32 --steps 24000 --root ./exp8 --device cu
3656663       07:43 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 13 --d 32 --steps 24000 --root ./exp8 --d
3656664       07:43 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 13 --d 32 --steps 24000 --root ./exp8 --device cu
3656942       07:30 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev4 --seed 13 --d 

In [ ]:
!ls exp8/logs | grep -E "_s7|_s13" | head

tr_identity_d32_s13.txt
tr_identity_d32_s7.txt
tr_marker_d32_s13.txt
tr_marker_d32_s7.txt
tr_perm_sym_d32_s13.txt
tr_perm_sym_d32_s7.txt
tr_relabel_new_d32_s13.txt
tr_relabel_new_d32_s7.txt
tr_relabel_perm_d32_s13.txt
tr_relabel_perm_d32_s7.txt


In [ ]:
!grep -l Traceback exp8/logs/tr_*_s7.txt exp8/logs/tr_*_s13.txt
!tail -n 1 exp8/logs/tr_rev4_d32_s7.txt

grep: exp8/logs/tr_*_s13.txt: No such file or directory


In [ ]:
!grep -l Traceback exp8/logs/tr_*_s7.txt
!tail -n 1 exp8/logs/tr_rev4_d32_s7.txt

In [9]:
write_prereg(ROOT)
r1 = check1(ROOT, d=D_MODEL, seeds=SEEDS)

already frozen: ./exp8/prereg_three_checks.txt
H-adapter pre-registration - written 2026-10-29 18:26
Written before ANY joint (s) result was read. Translator costs a(g) for seed 0 had already been seen.

GATE   mean gain_A' >= 0.02 nats and >= 3.0 SE. If it fails, every check below is UNTESTABLE, not negative.

CHECK 1 (dissociation) needs >= 3 seeds.
  PASS         s(rev_all) - max(s(rev4), s(rev45)) >= 0.1 in every seed
  FAIL         mean of that difference <= 0
  INCONCLUSIVE otherwise
  Secondary (reported, not part of the verdict): perm_sym is among the two lowest s in every seed.

CHECK 2 (a beats string distance) needs >= 8 renderings with finite a and s, and >= 3 seeds.
  PASS         partial rho(a, s | tau, displacement) <= -0.5
               AND rho(a, s) <= rho(tau, s) - 0.1
               AND the leave-one-rendering-out partial never exceeds -0.3
  FAIL         partial >= -0.2
  INCONCLUSIVE otherwise

CHECK 3 (depth) - see exp8_depth.py for its frozen thresholds.

=== CH

In [10]:
!ps -eo pid,etime,args | grep -E "exp8_queue|exp8_lib" | grep -v grep | cut -c1-120
!tail -n 3 exp8/scheduler.log

3519158    02:56:40 /home/anwoy/miniconda3/envs/ke/bin/python exp8_queue.py --root ./exp8 --d 32 --seeds 0 --devices cud
3656565       47:02 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev6 --seed 13 --d 32 -
3656566       47:02 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev6 --seed 13 --d 32 --steps 2
3656663       46:58 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 13 --d 32 -
3656664       46:58 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 13 --d 32 --steps 2
3656942       46:44 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev4 --seed 13 --d 32 -
3656943       46:44 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev4 --seed 13 --d 32 --steps 2
3657111       46:38 bash -c /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev45 --seed 13 --d 32 
3657112       46:38 /home/anwoy/

In [11]:
!for g in rev6 rev5 rev4 rev45 rev_all perm_sym; do echo "$g: $(tail -n 1 exp8/logs/tr_${g}_d32_s13.txt | cut -c1-100)"; done

tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 
tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 
tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 
tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 
tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 
tail: cannot open 'exp8/logs/tr_.txt' for reading: No such file or directory
perm_sym: 


In [13]:
import os
for g in ["rev6", "rev5", "rev4", "rev45", "rev_all", "perm_sym"]:
    p = f"exp8/logs/tr_{g}_d32_s13.txt"
    lines = open(p).read().strip().splitlines() if os.path.exists(p) else []
    print(f"{g:9s}", lines[-1][:100] if lines else "no log yet")

rev6      {"g": "rev6", "d": 32, "seed": 13, "steps": 24000, "auc": 0.03255192505013232, "final_train": 0.0029
rev5      {"g": "rev5", "d": 32, "seed": 13, "steps": 24000, "auc": 0.0649237719590698, "final_train": 0.01489
rev4      {"g": "rev4", "d": 32, "seed": 13, "steps": 24000, "auc": 0.1669280248588572, "final_train": 0.05999
rev45     {"g": "rev45", "d": 32, "seed": 13, "steps": 24000, "auc": 0.1377087260982177, "final_train": 0.0428
rev_all   {"g": "rev_all", "d": 32, "seed": 13, "steps": 24000, "auc": 0.036170762269845065, "final_train": 0.
perm_sym  {"g": "perm_sym", "d": 32, "seed": 13, "steps": 24000, "auc": 0.2877468285004919, "final_train": 0.1


In [14]:
!ps -eo args | grep "exp8_lib.py joint" | grep -v grep | wc -l
!tail -n 4 exp8/scheduler.log

0
21:32:12 started jt_rev_all_disjoint_d32_s0 on cuda:2 pid 3717767
21:32:12 started jt_perm_sym_shared_d32_s0 on cuda:2 pid 3717768
21:32:22 started jt_perm_sym_disjoint_d32_s0 on cuda:2 pid 3718581
all jobs finished


In [15]:
import os, glob, time
G = ["relabel_new","relabel_perm","marker","swap5","rev6","rev5","rev4","rev45","rev_all","perm_sym"]
print(time.strftime("now %H:%M"), "(the nine jobs started at 21:32)")
missing = [(g, c) for g in G for c in ("shared", "disjoint") if not os.path.exists(f"exp8/joint/{g}_{c}_d32_s0.json")]
print(20 - len(missing), "of 20 joint JSONs;", "missing:", missing)
for g, c in missing:
    p = f"exp8/logs/jt_{g}_{c}_d32_s0.txt"; L = open(p).read().strip().splitlines() if os.path.exists(p) else []
    print(f"{g}_{c}: {L[-1][:150] if L else 'no log'}")

now 23:45 (the nine jobs started at 21:32)
20 of 20 joint JSONs; missing: []
